In [ ]:
!pip install -q --upgrade pip
!pip install -q "transformers>=4.44.0,<5.0.0" "accelerate>=0.33.0" "peft>=0.12.0" \
    "bitsandbytes>=0.43.3" "datasets>=2.20.0" "sentencepiece>=0.2.0" "protobuf>=4.25.0" \
    "sacrebleu[ja]>=2.4.3" "evaluate>=0.4.2" "tokenizers>=0.19.1" \
    "einops>=0.8.0" "safetensors>=0.4.3" "tqdm>=4.66.0" "tabulate>=0.9.0" "psutil>=6.0.0"
!pip install -q "torch>=2.3.0" --index-url https://download.pytorch.org/whl/cu121 || true

In [ ]:
import os, re, gc, sys, json, math, time, random, logging, warnings, unicodedata, html, subprocess, shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from tabulate import tabulate
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("BITSANDBYTES_NOWELCOME", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "0")
os.environ.setdefault("DATASETS_VERBOSITY", "error")
for _noisy in ("transformers", "datasets", "huggingface_hub", "peft", "accelerate", "bitsandbytes"):
    logging.getLogger(_noisy).setLevel(logging.ERROR)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("hymt2-qlora")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
log.info(f"Device: {DEVICE} | bf16 supported: {BF16_OK} | compute dtype: {COMPUTE_DTYPE}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    log.info(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB | CC: {p.major}.{p.minor}")


In [ ]:
@dataclass
class CFG:
    model_name: str = "tencent/Hy-MT2-1.8B"
    output_dir: str = "/teamspace/studios/this_studio/hymt2_qlora_out"
    data_root: str = "/teamspace/studios/this_studio/tvsub"
    src_lang: str = "Chinese"
    tgt_lang: str = "English"
    max_src_len: int = 256
    max_tgt_len: int = 256
    max_total_len: int = 640
    train_samples: int = 60000
    val_samples: int = 500
    test_samples: int = 500
    batch_size: int = 4
    eval_batch_size: int = 4
    grad_accum: int = 8
    epochs: int = 3
    lr: float = 2e-4
    weight_decay: float = 0.0
    warmup_ratio: float = 0.03
    max_grad_norm: float = 1.0
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: Tuple[str, ...] = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
    save_every_steps: int = 500
    eval_gen_max_new: int = 160
    eval_gen_num_beams: int = 4
    early_stop_patience: int = 2
    early_stop_min_delta: float = 0.05
    log_every: int = 25
    resume_from: str = ""
cfg = CFG()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.data_root).parent.mkdir(parents=True, exist_ok=True)
log.info(f"Config: {cfg}")


In [ ]:
if not Path(cfg.data_root).exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/longyuewangdcu/tvsub.git", cfg.data_root], check=True)
else:
    log.info(f"TVSub already present at {cfg.data_root}")
train_zh = Path(cfg.data_root) / "data/processed/train/train.zh"
train_en = Path(cfg.data_root) / "data/processed/train/train.en"
dev_src  = Path(cfg.data_root) / "data/processed/dev/dev_src"
dev_ref  = Path(cfg.data_root) / "data/processed/dev/dev_trg.sgm"
test_src = Path(cfg.data_root) / "data/processed/test/test_src"
test_ref = Path(cfg.data_root) / "data/processed/test/test_trg.sgm"
for p in [train_zh, train_en, dev_src, dev_ref, test_src, test_ref]:
    assert p.exists(), f"Missing dataset file: {p}"
    log.info(f"OK  {p}  {p.stat().st_size/1e6:.2f} MB")


In [ ]:
_HTML_UNESC = re.compile(r"&(?:amp|apos|quot|lt|gt|#\d+);")
_MULTISPACE = re.compile(r"\s+")
_SPACE_BEFORE_PUNCT = re.compile(r"\s+([,.!?;:%\)\]\}»”’])")
_SPACE_AFTER_PUNCT  = re.compile(r"([\(\[\{«“‘])\s+")

def detokenize(text: str) -> str:
    if text is None: return ""
    t = text.replace("@-@", "-").replace("@@ ", "").replace(" @@", "")
    t = html.unescape(t)
    t = _HTML_UNESC.sub(lambda m: html.unescape(m.group(0)), t)
    t = _MULTISPACE.sub(" ", t).strip()
    t = _SPACE_BEFORE_PUNCT.sub(r"\1", t)
    t = _SPACE_AFTER_PUNCT.sub(r"\1", t)
    t = unicodedata.normalize("NFKC", t)
    return t

_SEG_RE = re.compile(r'<seg\s+id\s*=\s*"?(\d+)"?\s*>(.*?)</seg>', re.IGNORECASE | re.DOTALL)
_DOC_RE = re.compile(r'<DOC\s+docid\s*=\s*"([^"]+)"\s+sysid\s*=\s*"([^"]+)"[^>]*>(.*?)</DOC>', re.IGNORECASE | re.DOTALL)

def parse_sgm_multiref(path: Path) -> Dict[int, List[str]]:
    raw = Path(path).read_text(encoding="utf-8", errors="ignore")
    refs: Dict[int, List[str]] = {}
    docs = _DOC_RE.findall(raw)
    if not docs:
        for sid, text in _SEG_RE.findall(raw):
            refs.setdefault(int(sid), []).append(detokenize(text))
        return refs
    for _docid, _sysid, body in docs:
        for sid, text in _SEG_RE.findall(body):
            refs.setdefault(int(sid), []).append(detokenize(text))
    return refs

def read_lines(path: Path) -> List[str]:
    return Path(path).read_text(encoding="utf-8", errors="ignore").splitlines()

def build_train_pairs(zh_path: Path, en_path: Path, limit: int) -> List[Dict[str, str]]:
    zh_lines = read_lines(zh_path); en_lines = read_lines(en_path)
    assert len(zh_lines) == len(en_lines), f"Mismatch: {len(zh_lines)} vs {len(en_lines)}"
    pairs = []
    for zh, en in zip(zh_lines, en_lines):
        zh_c, en_c = detokenize(zh), detokenize(en)
        if not zh_c or not en_c: continue
        if len(zh_c) > cfg.max_src_len * 2 or len(en_c) > cfg.max_tgt_len * 2: continue
        r = len(en_c) / max(1, len(zh_c))
        if r < 0.3 or r > 4.0: continue
        pairs.append({"src": zh_c, "tgt": en_c})
    rng = random.Random(SEED); rng.shuffle(pairs)
    if limit and len(pairs) > limit: pairs = pairs[:limit]
    return pairs

def build_eval_pairs(src_path: Path, sgm_path: Path, limit: int) -> Tuple[List[Dict[str, Any]], List[List[str]]]:
    src_lines = [detokenize(x) for x in read_lines(src_path)]
    refs_by_id = parse_sgm_multiref(sgm_path)
    n = min(len(src_lines), max(refs_by_id.keys()) if refs_by_id else len(src_lines))
    examples, multi_refs = [], []
    for i in range(1, n + 1):
        src = src_lines[i - 1] if i - 1 < len(src_lines) else ""
        refs = refs_by_id.get(i, [])
        if not src or not refs: continue
        examples.append({"src": src, "tgt": refs[0], "id": i})
        multi_refs.append(refs)
        if limit and len(examples) >= limit: break
    return examples, multi_refs

train_pairs = build_train_pairs(train_zh, train_en, cfg.train_samples)
val_pairs, val_multi_refs = build_eval_pairs(dev_src, dev_ref, cfg.val_samples)
test_pairs, test_multi_refs = build_eval_pairs(test_src, test_ref, cfg.test_samples)
log.info(f"Train pairs: {len(train_pairs)} | Val pairs: {len(val_pairs)} | Test pairs: {len(test_pairs)}")
log.info("Sample TRAIN:")
for ex in train_pairs[:3]: log.info(f"  ZH: {ex['src']}\n  EN: {ex['tgt']}")
log.info("Sample VAL:")
for ex, refs in list(zip(val_pairs, val_multi_refs))[:3]:
    log.info(f"  ID {ex['id']} | ZH: {ex['src']} | REFS({len(refs)}): {refs}")


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_cfg,
    device_map={"": 0} if torch.cuda.is_available() else "auto",
    trust_remote_code=True,
    torch_dtype=COMPUTE_DTYPE,
    attn_implementation="sdpa",
)
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
lora_cfg = LoraConfig(
    r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
    bias="none", task_type="CAUSAL_LM",
    target_modules=list(cfg.lora_target_modules),
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()
log.info(f"Model loaded on {next(model.parameters()).device} | dtype: {next(model.parameters()).dtype}")


In [ ]:
PROMPT_TEMPLATE = "Translate the following text into {tgt_lang}. Note that you must ONLY output the translated result without any additional explanation:\n\n{src}"

def build_input_ids(src: str, tgt: Optional[str]) -> Dict[str, List[int]]:
    user_content = PROMPT_TEMPLATE.format(tgt_lang=cfg.tgt_lang, src=src)
    msgs = [{"role": "user", "content": user_content}]
    prompt_ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True)
    if tgt is None:
        return {"input_ids": prompt_ids, "labels": None, "prompt_len": len(prompt_ids)}
    tgt_ids = tokenizer(tgt, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    input_ids = prompt_ids + tgt_ids
    if len(input_ids) > cfg.max_total_len:
        overflow = len(input_ids) - cfg.max_total_len
        tgt_ids = tgt_ids[: max(1, len(tgt_ids) - overflow)]
        input_ids = prompt_ids + tgt_ids
    labels = [-100] * len(prompt_ids) + list(tgt_ids)
    return {"input_ids": input_ids, "labels": labels, "prompt_len": len(prompt_ids)}

class MTDataset(Dataset):
    def __init__(self, pairs, is_train: bool):
        self.pairs = pairs; self.is_train = is_train
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        p = self.pairs[i]
        enc = build_input_ids(p["src"], p["tgt"] if self.is_train else None)
        enc["src"] = p["src"]; enc["tgt"] = p.get("tgt", "")
        return enc

def collate_train(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    ids = torch.full((len(batch), maxlen), pad, dtype=torch.long)
    attn = torch.zeros((len(batch), maxlen), dtype=torch.long)
    lbl = torch.full((len(batch), maxlen), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L = len(b["input_ids"])
        ids[i, :L]  = torch.tensor(b["input_ids"], dtype=torch.long)
        attn[i, :L] = 1
        lbl[i, :L]  = torch.tensor(b["labels"], dtype=torch.long)
    return {"input_ids": ids, "attention_mask": attn, "labels": lbl}

def collate_eval(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    ids = torch.full((len(batch), maxlen), pad, dtype=torch.long)
    attn = torch.zeros((len(batch), maxlen), dtype=torch.long)
    for i, b in enumerate(batch):
        L = len(b["input_ids"])
        pad_left = maxlen - L
        ids[i, pad_left:]  = torch.tensor(b["input_ids"], dtype=torch.long)
        attn[i, pad_left:] = 1
    return {"input_ids": ids, "attention_mask": attn,
            "srcs": [b["src"] for b in batch], "tgts": [b["tgt"] for b in batch]}

train_ds = MTDataset(train_pairs, is_train=True)
val_ds   = MTDataset(val_pairs,   is_train=False)
test_ds  = MTDataset(test_pairs,  is_train=False)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  collate_fn=collate_train, num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.eval_batch_size, shuffle=False, collate_fn=collate_eval,  num_workers=2, pin_memory=True)
log.info(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


In [ ]:
def gpu_mem_summary(tag: str = ""):
    if not torch.cuda.is_available(): return
    a = torch.cuda.memory_allocated() / 1e9
    r = torch.cuda.memory_reserved() / 1e9
    m = torch.cuda.max_memory_allocated() / 1e9
    log.info(f"[MEM {tag}] allocated={a:.2f}GB reserved={r:.2f}GB peak={m:.2f}GB")

def check_dataset_integrity(ds, name, n=3):
    log.info(f"Dataset check: {name} (size={len(ds)}) ---")
    for i in range(min(n, len(ds))):
        item = ds[i]
        ids = item["input_ids"]; lbl = item.get("labels")
        decoded_prompt = tokenizer.decode(ids[:item["prompt_len"]], skip_special_tokens=False)[:200]
        log.info(f"  [{i}] len(ids)={len(ids)} prompt_len={item['prompt_len']}")
        log.info(f"       prompt_head: {decoded_prompt}")
        if lbl is not None:
            n_train = sum(1 for x in lbl if x != -100)
            log.info(f"       supervised_tokens={n_train} / {len(lbl)}")

def tokenizer_sanity():
    s = "抱歉，怎么称呼？"
    ids = tokenizer(s, return_tensors="pt")["input_ids"]
    log.info(f"[TOK] '{s}' -> {ids.shape[-1]} tokens | roundtrip='{tokenizer.decode(ids[0], skip_special_tokens=True)}'")
    log.info(f"[TOK] pad={tokenizer.pad_token_id} eos={tokenizer.eos_token_id} bos={tokenizer.bos_token_id}")

def contains_nan_or_inf(t: torch.Tensor) -> bool:
    return bool(torch.isnan(t).any().item() or torch.isinf(t).any().item())

def grad_norm(m) -> float:
    total = 0.0
    for p in m.parameters():
        if p.grad is not None:
            total += p.grad.detach().float().pow(2).sum().item()
    return math.sqrt(total)

tokenizer_sanity()
check_dataset_integrity(train_ds, "train")
check_dataset_integrity(val_ds,   "val")
gpu_mem_summary("after-model-load")


In [ ]:
from torch.optim import AdamW
from torch.amp import GradScaler
try:
    from bitsandbytes.optim import PagedAdamW8bit
    optim_cls = PagedAdamW8bit
    log.info("Using bitsandbytes PagedAdamW8bit")
except Exception:
    optim_cls = AdamW
    log.info("Falling back to torch AdamW")

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = optim_cls(trainable, lr=cfg.lr, weight_decay=cfg.weight_decay, betas=(0.9, 0.999), eps=1e-8)
total_update_steps = math.ceil(len(train_loader) / cfg.grad_accum) * cfg.epochs
warmup_steps = max(1, int(cfg.warmup_ratio * total_update_steps))

def cosine_lr(step, total, warmup, base_lr):
    if step < warmup: return base_lr * step / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

use_amp_fp16 = (not BF16_OK) and torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp_fp16)
autocast_dtype = torch.bfloat16 if BF16_OK else torch.float16
log.info(f"total_update_steps={total_update_steps} warmup_steps={warmup_steps} amp_fp16_scaler={use_amp_fp16}")
from torch.optim import AdamW
try:
    from bitsandbytes.optim import PagedAdamW8bit
    optim_cls = PagedAdamW8bit
    log.info("Using bitsandbytes PagedAdamW8bit")
except Exception:
    optim_cls = AdamW
    log.info("Falling back to torch AdamW")

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = optim_cls(trainable, lr=cfg.lr, weight_decay=cfg.weight_decay, betas=(0.9, 0.999), eps=1e-8)
total_update_steps = math.ceil(len(train_loader) / cfg.grad_accum) * cfg.epochs
warmup_steps = max(1, int(cfg.warmup_ratio * total_update_steps))

def cosine_lr(step, total, warmup, base_lr):
    if step < warmup: return base_lr * step / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

use_amp_fp16 = (not BF16_OK) and torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp_fp16)
autocast_dtype = torch.bfloat16 if BF16_OK else torch.float16
log.info(f"total_update_steps={total_update_steps} warmup_steps={warmup_steps} amp_fp16_scaler={use_amp_fp16}")

In [ ]:
import sacrebleu
try:
    import evaluate
    bleu_hf = evaluate.load("bleu")
    HAVE_HF_BLEU = True
except Exception as e:
    HAVE_HF_BLEU = False; log.warning(f"HF evaluate BLEU unavailable: {e}")

def compute_metrics(hyps: List[str], refs_single: List[str], refs_multi: List[List[str]], srcs: List[str]) -> Dict[str, float]:
    out = {}
    max_r = max(len(r) for r in refs_multi) if refs_multi else 1
    padded = [(r + [r[-1]] * (max_r - len(r))) if r else [""] * max_r for r in refs_multi]
    ref_lists = list(map(list, zip(*padded))) if padded else [refs_single]
    sb = sacrebleu.corpus_bleu(hyps, ref_lists, tokenize="13a", smooth_method="exp")
    out["sacreBLEU"] = float(sb.score)
    chrf = sacrebleu.corpus_chrf(hyps, ref_lists)
    out["chrF"] = float(chrf.score)
    try:
        ter = sacrebleu.corpus_ter(hyps, ref_lists)
        out["TER"] = float(ter.score)
    except Exception:
        out["TER"] = float("nan")
    if HAVE_HF_BLEU:
        try:
            b = bleu_hf.compute(predictions=hyps, references=refs_multi if refs_multi else [[r] for r in refs_single])
            out["BLEU"] = float(b["bleu"]) * 100.0
        except Exception:
            out["BLEU"] = float("nan")
    else:
        out["BLEU"] = float("nan")
    return out


In [ ]:
@torch.no_grad()
def evaluate_val_loss(m, loader) -> float:
    m.eval(); losses = []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE); attn = batch["attention_mask"].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=autocast_dtype, enabled=torch.cuda.is_available()):
            lbl = build_teacher_labels(batch, m)
            out = m(input_ids=input_ids, attention_mask=attn, labels=lbl)
        losses.append(out.loss.item())
    return float(np.mean(losses)) if losses else float("nan")

def build_teacher_labels(batch, m):
    return batch["input_ids"].to(DEVICE).clone().masked_fill(batch["attention_mask"].to(DEVICE) == 0, -100)

@torch.no_grad()
def generate_and_score(m, loader, multi_refs: List[List[str]]) -> Tuple[Dict[str, float], List[str]]:
    m.eval(); hyps, srcs, tgts = [], [], []
    for batch in tqdm(loader, desc="gen", leave=False):
        input_ids = batch["input_ids"].to(DEVICE); attn = batch["attention_mask"].to(DEVICE)
        gen = m.generate(
            input_ids=input_ids, attention_mask=attn,
            max_new_tokens=cfg.eval_gen_max_new,
            num_beams=cfg.eval_gen_num_beams,
            do_sample=False, temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
        new_tokens = gen[:, input_ids.shape[1]:]
        for row in new_tokens:
            txt = tokenizer.decode(row, skip_special_tokens=True).strip()
            hyps.append(txt)
        srcs.extend(batch["srcs"]); tgts.extend(batch["tgts"])
    refs_multi = multi_refs[: len(hyps)] if multi_refs else [[t] for t in tgts]
    metrics = compute_metrics(hyps, tgts, refs_multi, srcs)
    return metrics, hyps

def print_metrics_table(epoch: int, val_loss: float, metrics: Dict[str, float]):
    rows = [["epoch", epoch], ["val_loss", f"{val_loss:.4f}"]]
    for k in ["BLEU", "sacreBLEU", "chrF", "TER"]:
        v = metrics.get(k, float("nan"))
        rows.append([k, f"{v:.2f}" if not math.isnan(v) else "n/a"])
    print("\n" + tabulate(rows, headers=["metric", "value"], tablefmt="github") + "\n")


In [ ]:
CKPT_DIR = Path(cfg.output_dir) / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_DIR = Path(cfg.output_dir) / "best"; BEST_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE = Path(cfg.output_dir) / "trainer_state.json"

def save_checkpoint(m, opt, step: int, epoch: int, best_score: float, tag: str):
    tgt = CKPT_DIR / f"{tag}"
    tgt.mkdir(parents=True, exist_ok=True)
    m.save_pretrained(str(tgt))
    tokenizer.save_pretrained(str(tgt))
    torch.save({"optimizer": opt.state_dict(), "step": step, "epoch": epoch,
                "best_score": best_score, "rng_torch": torch.get_rng_state(),
                "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
               tgt / "trainer.pt")
    STATE_FILE.write_text(json.dumps({"last_ckpt": str(tgt), "step": step, "epoch": epoch, "best_score": best_score}))
    log.info(f"[CKPT] saved -> {tgt}")

def save_best(m, score: float):
    for p in BEST_DIR.glob("*"):
        if p.is_file(): p.unlink()
        else: shutil.rmtree(p)
    m.save_pretrained(str(BEST_DIR)); tokenizer.save_pretrained(str(BEST_DIR))
    (BEST_DIR / "score.json").write_text(json.dumps({"best_score": score}))
    log.info(f"[BEST] score={score:.4f} -> {BEST_DIR}")

def try_resume(m, opt) -> Tuple[int, int, float]:
    resume_path = cfg.resume_from or (json.loads(STATE_FILE.read_text())["last_ckpt"] if STATE_FILE.exists() else "")
    if not resume_path or not Path(resume_path).exists():
        return 0, 0, -float("inf")
    log.info(f"[RESUME] from {resume_path}")
    adapter = PeftModel.from_pretrained(m.get_base_model(), resume_path, is_trainable=True)
    for n, p in adapter.named_parameters():
        if n in dict(m.named_parameters()):
            dict(m.named_parameters())[n].data.copy_(p.data)
    tr = torch.load(Path(resume_path) / "trainer.pt", map_location="cpu")
    try: opt.load_state_dict(tr["optimizer"])
    except Exception as e: log.warning(f"optimizer state not restored: {e}")
    if tr.get("rng_torch") is not None: torch.set_rng_state(tr["rng_torch"])
    if tr.get("rng_cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(tr["rng_cuda"])
    return int(tr.get("step", 0)), int(tr.get("epoch", 0)), float(tr.get("best_score", -float("inf")))

start_step, start_epoch, best_score = try_resume(model, optimizer)
log.info(f"start_step={start_step} start_epoch={start_epoch} best_score={best_score}")


In [ ]:
def set_lr(step):
    lr = cosine_lr(step, total_update_steps, warmup_steps, cfg.lr)
    for g in optimizer.param_groups: g["lr"] = lr
    return lr

global_step = start_step; update_step = start_step
patience_left = cfg.early_stop_patience
stop_training = False
t_train_start = time.time()

for epoch in range(start_epoch, cfg.epochs):
    if stop_training: break
    model.train(); optimizer.zero_grad(set_to_none=True)
    t_epoch = time.time(); running = {"loss": 0.0, "n": 0, "toks": 0}
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"epoch {epoch+1}/{cfg.epochs}")
    for it, batch in pbar:
        try:
            input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            attn      = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labels    = batch["labels"].to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=autocast_dtype, enabled=torch.cuda.is_available()):
                out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
                loss = out.loss / cfg.grad_accum
            if contains_nan_or_inf(loss.detach()):
                log.warning(f"[NaN] loss at step {global_step}; skipping batch"); optimizer.zero_grad(set_to_none=True); continue
            if use_amp_fp16: scaler.scale(loss).backward()
            else: loss.backward()
            running["loss"] += float(loss.item()) * cfg.grad_accum
            running["n"]    += 1
            running["toks"] += int(attn.sum().item())
            do_step = ((it + 1) % cfg.grad_accum == 0) or (it + 1 == len(train_loader))
            if do_step:
                if use_amp_fp16:
                    scaler.unscale_(optimizer)
                gn = torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm).item()
                lr_now = set_lr(update_step)
                if use_amp_fp16:
                    scaler.step(optimizer); scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                update_step += 1
                if update_step % cfg.log_every == 0:
                    avg_loss = running["loss"] / max(1, running["n"])
                    tps = running["toks"] / max(1e-9, (time.time() - t_epoch))
                    pbar.set_postfix(loss=f"{avg_loss:.4f}", lr=f"{lr_now:.2e}", gn=f"{gn:.2f}", tok_s=f"{tps:.0f}")
                    if update_step % (cfg.log_every * 4) == 0: gpu_mem_summary(f"step-{update_step}")
                if update_step % cfg.save_every_steps == 0:
                    save_checkpoint(model, optimizer, update_step, epoch, best_score, tag=f"step-{update_step}")
            global_step += 1
        except torch.cuda.OutOfMemoryError as e:
            log.error(f"OOM at step {global_step}: {e}"); torch.cuda.empty_cache(); gc.collect(); optimizer.zero_grad(set_to_none=True); continue
        except Exception as e:
            log.exception(f"training exception at step {global_step}: {e}"); optimizer.zero_grad(set_to_none=True); continue

    val_loss = evaluate_val_loss(model, val_loader)
    val_metrics, _ = generate_and_score(model, val_loader, val_multi_refs)
    print_metrics_table(epoch + 1, val_loss, val_metrics)
    save_checkpoint(model, optimizer, update_step, epoch + 1, best_score, tag=f"epoch-{epoch+1}")
    primary = val_metrics.get("sacreBLEU", -float("inf"))
    if primary > best_score + cfg.early_stop_min_delta:
        best_score = primary; patience_left = cfg.early_stop_patience; save_best(model, best_score)
    else:
        patience_left -= 1
        log.info(f"[EARLY-STOP] no improvement (patience_left={patience_left}, best={best_score:.4f}, current={primary:.4f})")
        if patience_left <= 0:
            log.info("[EARLY-STOP] stopping training"); stop_training = True
    log.info(f"[EPOCH {epoch+1}] time={ (time.time()-t_epoch)/60:.2f} min")

log.info(f"[TRAIN DONE] total_time={ (time.time()-t_train_start)/60:.2f} min | best sacreBLEU={best_score:.4f}")


In [ ]:
if (BEST_DIR / "adapter_config.json").exists():
    log.info(f"Loading best adapter from {BEST_DIR} for final test-set evaluation")
    del model; gc.collect(); torch.cuda.empty_cache()
    base_reload = AutoModelForCausalLM.from_pretrained(
        cfg.model_name, quantization_config=bnb_cfg,
        device_map={"": 0} if torch.cuda.is_available() else "auto",
        trust_remote_code=True, torch_dtype=COMPUTE_DTYPE, attn_implementation="sdpa",
    )
    base_reload.config.use_cache = True
    best_model = PeftModel.from_pretrained(base_reload, str(BEST_DIR))
    best_model.eval()
    test_loader = DataLoader(test_ds, batch_size=cfg.eval_batch_size, shuffle=False,
                             collate_fn=collate_eval, num_workers=2, pin_memory=True)
    test_metrics, test_hyps = generate_and_score(best_model, test_loader, test_multi_refs)
    print("\n Final Test Set:")
    print_metrics_table("TEST", float("nan"), test_metrics)
    out_file = Path(cfg.output_dir) / "test_hypotheses.txt"
    out_file.write_text("\n".join(test_hyps), encoding="utf-8")
    log.info(f"Wrote {len(test_hyps)} test hypotheses to {out_file}")
else:
    log.warning("No best checkpoint found; skipping final test evaluation.")


In [ ]:
def translate_one(m, src: str) -> str:
    m.eval()
    enc = build_input_ids(src, tgt=None)
    input_ids = torch.tensor([enc["input_ids"]], device=DEVICE)
    attn = torch.ones_like(input_ids)
    with torch.no_grad():
        gen = m.generate(input_ids=input_ids, attention_mask=attn,
                         max_new_tokens=cfg.eval_gen_max_new, num_beams=cfg.eval_gen_num_beams,
                         do_sample=False, pad_token_id=tokenizer.pad_token_id,
                         eos_token_id=tokenizer.eos_token_id, use_cache=True)
    return tokenizer.decode(gen[0, input_ids.shape[1]:], skip_special_tokens=True).strip()

_demo_model = locals().get("best_model", locals().get("model"))
for s in ["抱歉 怎么 称呼", "我 冒犯 你 了 吗", "今天天气真好，我们一起去公园散步吧。"]:
    print(f"SRC: {s}\nMT : {translate_one(_demo_model, detokenize(s))}\n")
